# PyTorch neural networks

In the previous exercise, we progressed from NumPy to PyTorch's `nn.Sequential` and `optim` abstractions. Now we take the final step: defining **custom neural networks** by subclassing `nn.Module`.

This is the standard way to build models in PyTorch and the pattern you will use in tomorrow's workshop.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Part A: Regression — fitting a noisy sine wave

### The data

We generate noisy samples from $y = \sin(x)$.

In [ ]:
torch.manual_seed(42)

N = 500
x_train = torch.linspace(-2 * np.pi, 2 * np.pi, N).unsqueeze(1)  # shape: (N, 1)
y_train = torch.sin(x_train) + 0.2 * torch.randn(N, 1)           # noisy sine

plt.scatter(x_train.numpy(), y_train.numpy(), s=5, alpha=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Training data: noisy sine wave")
plt.show()

### Exercise 1: Define the network

Create a class `SineNet` that inherits from `nn.Module`. The network should have:

- Input: 1 feature ($x$)
- Hidden layer 1: 32 neurons + ReLU activation
- Hidden layer 2: 32 neurons + ReLU activation
- Output: 1 value ($\hat{y}$)

Remember the pattern from the classes exercise:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()           # always call this first
        self.layer1 = nn.Linear(...)  # define layers as attributes
        self.relu = nn.ReLU()

    def forward(self, x):            # define the computation
        x = self.relu(self.layer1(x))
        return x
```

In [ ]:
class SineNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define layers

    def forward(self, x):
        # TODO: define forward pass
        raise NotImplementedError

### Exercise 2: Train the network

Write the training loop. Use:
- Loss function: `nn.MSELoss()`
- Optimizer: `torch.optim.Adam(model.parameters(), lr=1e-3)`

The training loop follows the 5-line pattern:
1. Forward pass: `y_pred = model(x_train)`
2. Compute loss: `loss = loss_fn(y_pred, y_train)`
3. Zero gradients: `optimizer.zero_grad()`
4. Backward pass: `loss.backward()`
5. Update weights: `optimizer.step()`

Train for 2000 epochs and record the loss.

In [ ]:
model = SineNet()
loss_fn = ...      # TODO
optimizer = ...    # TODO

losses = []
for epoch in range(2000):
    # TODO: training loop
    pass

### Exercise 3: Visualize the results

Plot:
1. The training loss over epochs
2. The model predictions vs the true sine function

In [ ]:
# TODO: plot loss curve and model fit


---

## Part B: Classification — two moons

### The data

We generate a 2D classification dataset: two interleaved half-circles ("moons"). This is a classic nonlinear classification problem.

In [ ]:
def make_moons(n_samples=500, noise=0.1, seed=42):
    """Generate two interleaved half-circles."""
    rng = np.random.default_rng(seed)
    n_half = n_samples // 2

    # Upper moon
    theta1 = np.linspace(0, np.pi, n_half)
    x1 = np.column_stack([np.cos(theta1), np.sin(theta1)])

    # Lower moon (shifted)
    theta2 = np.linspace(0, np.pi, n_samples - n_half)
    x2 = np.column_stack([1 - np.cos(theta2), 1 - np.sin(theta2) - 0.5])

    X = np.vstack([x1, x2]) + noise * rng.standard_normal((n_samples, 2))
    y = np.hstack([np.zeros(n_half), np.ones(n_samples - n_half)])

    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)


X_train, y_train = make_moons(n_samples=500, noise=0.15)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", s=10, alpha=0.7)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Two moons dataset")
plt.show()

### Exercise 4: Define a classifier

Create a `MoonClassifier` class that inherits from `nn.Module`:

- Input: 2 features ($x_1$, $x_2$)
- Hidden layer 1: 32 neurons + ReLU
- Hidden layer 2: 16 neurons + ReLU
- Output: 1 value + **Sigmoid** activation (to produce a probability between 0 and 1)

In [ ]:
class MoonClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define layers

    def forward(self, x):
        # TODO: define forward pass
        raise NotImplementedError

### Exercise 5: Train the classifier

Use:
- Loss function: `nn.BCELoss()` (Binary Cross-Entropy — the standard loss for binary classification with sigmoid output)
- Optimizer: `torch.optim.Adam(model.parameters(), lr=1e-2)`

Train for 1000 epochs.

In [ ]:
model = MoonClassifier()
loss_fn = ...      # TODO
optimizer = ...    # TODO

losses = []
for epoch in range(1000):
    # TODO: training loop
    pass

### Exercise 6: Plot the decision boundary

To visualize the decision boundary, evaluate the model on a grid of $(x_1, x_2)$ values and plot a contour where the predicted probability crosses 0.5.

The helper function below handles the grid evaluation — just pass in your trained model.

In [ ]:
def plot_classification_result(model, X_train, y_train, losses):
    """Plot loss curve and decision boundary."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Loss curve
    ax1.plot(losses)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training loss")

    # Decision boundary
    x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
    y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    with torch.no_grad():
        probs = model(grid).numpy().reshape(xx.shape)

    ax2.contourf(xx, yy, probs, levels=50, cmap="RdBu_r", alpha=0.6)
    ax2.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=2)
    ax2.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", s=10, edgecolors="k", linewidth=0.3)
    ax2.set_xlabel("$x_1$")
    ax2.set_ylabel("$x_2$")
    ax2.set_title("Decision boundary")

    plt.tight_layout()
    plt.show()


plot_classification_result(model, X_train, y_train, losses)

### Experiment

Try modifying the classifier and re-training:

- What happens if you remove one hidden layer?
- What happens if you use only 4 neurons per layer?
- What happens if you replace ReLU with `nn.Tanh()`?
- What happens if you use `torch.optim.SGD` instead of Adam?

In [ ]:
# Try your experiments here
